# Análisis de clientes con Spark — Entregable 2

**Caso de negocio:** cadena de comercio — desempeño por departamento, top categorías por ciudad, cliente estrella por ciudad y evolución mensual de ventas.

**Integrantes:** Pedro Sierra y Tomás Lopera

**Datos:** `transacciones.csv` (20.000 filas: id_tx, id_cliente, id_ciudad, categoria, monto, fecha) y `ciudades.csv` (7 filas: id_ciudad, ciudad, departamento).

> Se utiliza Databricks para establecer la sesión de Spark

## Carga de datos

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F
import time

df = spark.table("analisis_clientes.default.transacciones")
ciu = spark.table("analisis_clientes.default.ciudades")

df.createOrReplaceTempView("transacciones")
ciu.createOrReplaceTempView("ciudades")

print("transacciones:", df.count(), "filas")
print("ciudades:", ciu.count(), "filas")
df.printSchema()
ciu.printSchema()

transacciones: 20000 filas
ciudades: 7 filas
root
 |-- id_tx: long (nullable = true)
 |-- id_cliente: long (nullable = true)
 |-- id_ciudad: long (nullable = true)
 |-- categoria: string (nullable = true)
 |-- monto: double (nullable = true)
 |-- fecha: date (nullable = true)

root
 |-- id_ciudad: long (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- departamento: string (nullable = true)



In [0]:
display(df)

id_tx,id_cliente,id_ciudad,categoria,monto,fecha
1,726,2,mercado,56.2,2025-08-10
2,157,2,hogar,72.73,2025-06-21
3,41,1,electronica,89.46,2025-06-15
4,544,3,electronica,64.41,2025-05-27
5,480,1,electronica,220.36,2025-06-24
6,259,1,deportes,22.59,2025-01-06
7,412,5,mercado,68.6,2025-03-05
8,60,1,electronica,209.38,2025-12-04
9,294,1,ropa,13.65,2025-02-27
10,125,3,electronica,616.55,2025-02-20


In [0]:
display(ciu)

id_ciudad,ciudad,departamento
1,Medellin,Antioquia
2,Bogota,Cundinamarca
3,Cali,Valle
4,Barranquilla,Atlantico
5,Bucaramanga,Santander
6,Cartagena,Bolivar
7,Pereira,Risaralda


## Desempeño por departamento

¿Cómo se desempeña cada departamento? Ventas totales, ticket promedio y número de ventas.

In [0]:
t1 = (
    df.join(ciu, "id_ciudad")
      .groupBy("departamento")
      .agg(
          F.sum("monto").alias("ventas_totales"),
          F.avg("monto").alias("ticket_promedio"),
          F.count("*").alias("numero_ventas"),
      )
      .orderBy(F.desc("ventas_totales"))
)
t1.show(truncate=False)

+------------+------------------+------------------+-------------+
|departamento|ventas_totales    |ticket_promedio   |numero_ventas|
+------------+------------------+------------------+-------------+
|Cundinamarca|975402.2300000009 |137.24528352328704|7107         |
|Antioquia   |825275.5599999996 |138.79508240834167|5946         |
|Valle       |419686.4799999993 |140.1290417362268 |2995         |
|Atlantico   |231001.20999999976|138.40695626123411|1669         |
|Santander   |141531.31000000003|150.08622481442208|943          |
|Bolivar     |112403.37000000014|146.16823146944103|769          |
|Risaralda   |80148.86999999997 |140.36579684763566|571          |
+------------+------------------+------------------+-------------+



In [0]:
t1_sql = spark.sql("""
    SELECT c.departamento,
           SUM(t.monto)   AS ventas_totales,
           AVG(t.monto)   AS ticket_promedio,
           COUNT(*)       AS numero_ventas
    FROM transacciones t
    JOIN ciudades c ON t.id_ciudad = c.id_ciudad
    GROUP BY c.departamento
    ORDER BY ventas_totales DESC
""")
t1_sql.show(truncate=False)

+------------+------------------+------------------+-------------+
|departamento|ventas_totales    |ticket_promedio   |numero_ventas|
+------------+------------------+------------------+-------------+
|Cundinamarca|975402.2300000009 |137.24528352328704|7107         |
|Antioquia   |825275.5599999996 |138.79508240834167|5946         |
|Valle       |419686.4799999993 |140.1290417362268 |2995         |
|Atlantico   |231001.20999999976|138.40695626123411|1669         |
|Santander   |141531.31000000003|150.08622481442208|943          |
|Bolivar     |112403.37000000014|146.16823146944103|769          |
|Risaralda   |80148.86999999997 |140.36579684763566|571          |
+------------+------------------+------------------+-------------+



In [0]:
total_general = df.agg(F.sum("monto")).collect()[0][0]
suma_departamentos = t1.agg(F.sum("ventas_totales")).collect()[0][0]
print(f"Suma por departamentos: {suma_departamentos:,.2f}")
print(f"Total general:          {total_general:,.2f}")
assert round(suma_departamentos, 2) == round(total_general, 2), "¡No cierra! revisar el JOIN"

Suma por departamentos: 2,785,449.03
Total general:          2,785,449.03


### Resultados

| departamento | ventas_totales | ticket_promedio | numero_ventas |
|---|---:|---:|---:|
| Cundinamarca | 975,402.23 | 137.25 | 7,107 |
| Antioquia | 825,275.56 | 138.80 | 5,946 |
| Valle | 419,686.48 | 140.13 | 2,995 |
| Atlantico | 231,001.21 | 138.41 | 1,669 |
| Santander | 141,531.31 | 150.09 | 943 |
| Bolivar | 112,403.37 | 146.17 | 769 |
| Risaralda | 80,148.87 | 140.37 | 571 |

La suma de las 7 filas es **2,785,449.03**, idéntica al total general de transacciones: el JOIN está bien construido y no se está perdiendo ni duplicando información.

### Análisis

- **¿Qué departamento vende más?** Cundinamarca (Bogotá), con **$975.402**, seguido de cerca por Antioquia (Medellín) con **$825.276**. Entre los dos concentran **65% de las ventas totales** del año — son, con diferencia, los mercados más importantes de la cadena.

- La diferencia entre el 1° y el 2° lugar (~18%) es mucho menor que la diferencia entre el 2° y el 3° lugar (Valle, con solo **$419.686**, menos de la mitad que Antioquia). Esto sugiere que el negocio tiene **dos plazas ancla claras** (Bogotá y Medellín) y un segundo grupo de ciudades intermedias (Cali, Barranquilla) muy por debajo.

- El número de ventas (`numero_ventas`) es casi proporcional a las ventas totales entre departamentos — es decir, la diferencia en ingresos se explica principalmente por **volumen de transacciones**, no por tickets más altos en las plazas grandes.

- **¿El ticket promedio varía mucho entre regiones?** No. Va de **$137.25** (Cundinamarca) a **$150.09** (Santander), una dispersión de apenas **~9.4%**. Esto es un hallazgo relevante: el comportamiento de compra por transacción es *homogéneo* en todo el país — no hay una región donde la gente gaste sistemáticamente más o menos por compra. Curiosamente, las regiones con menor volumen (Santander, Bolívar) tienen el ticket promedio más alto, aunque la diferencia no es económicamente significativa.

- **Implicación de negocio:** la prioridad de inversión/logística debería ir donde está el volumen (Cundinamarca y Antioquia), no donde está el ticket promedio más alto, ya que este último varía poco. Las plazas pequeñas (Risaralda, Bolívar) tienen espacio de crecimiento por *volumen*, no por subir el ticket.

## Top 3 categorías por ciudad

¿Cuáles son las 3 categorías que más venden en cada ciudad? 

> Se usa `ROW_NUMBER()` sobre una ventana particionada por ciudad porque un `GROUP BY` simple no permite quedarse con el top-N *dentro de cada grupo* conservando el orden/ranking.

In [0]:
ventas_ciudad_cat = (
    df.join(ciu, "id_ciudad")
      .groupBy("ciudad", "categoria")
      .agg(F.sum("monto").alias("ventas_categoria"))
)

w_ciudad = Window.partitionBy("ciudad").orderBy(F.desc("ventas_categoria"))

t2 = (
    ventas_ciudad_cat
      .withColumn("rank", F.row_number().over(w_ciudad))
      .filter(F.col("rank") <= 3)
      .orderBy("ciudad", "rank")
)
t2.show(30, truncate=False)

+------------+-----------+------------------+----+
|ciudad      |categoria  |ventas_categoria  |rank|
+------------+-----------+------------------+----+
|Barranquilla|electronica|123166.10000000008|1   |
|Barranquilla|ropa       |33464.20000000001 |2   |
|Barranquilla|hogar      |32475.319999999992|3   |
|Bogota      |electronica|525325.1999999991 |1   |
|Bogota      |hogar      |134266.17000000016|2   |
|Bogota      |ropa       |115588.88000000008|3   |
|Bucaramanga |electronica|84621.34          |1   |
|Bucaramanga |hogar      |16809.120000000003|2   |
|Bucaramanga |ropa       |16009.739999999996|3   |
|Cali        |electronica|223732.21999999997|1   |
|Cali        |hogar      |60514.649999999994|2   |
|Cali        |ropa       |48702.24999999998 |3   |
|Cartagena   |electronica|67209.05999999998 |1   |
|Cartagena   |hogar      |13175.33          |2   |
|Cartagena   |ropa       |10991.369999999999|3   |
|Medellin    |electronica|447357.7499999996 |1   |
|Medellin    |hogar      |11663

In [0]:
t2_sql = spark.sql("""
    WITH ventas_ciudad_cat AS (
        SELECT c.ciudad, t.categoria, SUM(t.monto) AS ventas_categoria
        FROM transacciones t
        JOIN ciudades c ON t.id_ciudad = c.id_ciudad
        GROUP BY c.ciudad, t.categoria
    ),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY ciudad ORDER BY ventas_categoria DESC) AS rank
        FROM ventas_ciudad_cat
    )
    SELECT ciudad, categoria, ventas_categoria, rank
    FROM ranked
    WHERE rank <= 3
    ORDER BY ciudad, rank
""")
t2_sql.show(30, truncate=False)

+------------+-----------+------------------+----+
|ciudad      |categoria  |ventas_categoria  |rank|
+------------+-----------+------------------+----+
|Barranquilla|electronica|123166.10000000008|1   |
|Barranquilla|ropa       |33464.20000000001 |2   |
|Barranquilla|hogar      |32475.319999999992|3   |
|Bogota      |electronica|525325.1999999991 |1   |
|Bogota      |hogar      |134266.17000000016|2   |
|Bogota      |ropa       |115588.88000000008|3   |
|Bucaramanga |electronica|84621.34          |1   |
|Bucaramanga |hogar      |16809.120000000003|2   |
|Bucaramanga |ropa       |16009.739999999996|3   |
|Cali        |electronica|223732.21999999997|1   |
|Cali        |hogar      |60514.649999999994|2   |
|Cali        |ropa       |48702.24999999998 |3   |
|Cartagena   |electronica|67209.05999999998 |1   |
|Cartagena   |hogar      |13175.33          |2   |
|Cartagena   |ropa       |10991.369999999999|3   |
|Medellin    |electronica|447357.7499999996 |1   |
|Medellin    |hogar      |11663

### Resultados

| ciudad | #1 | #2 | #3 |
|---|---|---|---|
| Bogota | electronica (525,325) | hogar (134,266) | ropa (115,589) |
| Medellin | electronica (447,358) | hogar (116,633) | ropa (101,595) |
| Cali | electronica (223,732) | hogar (60,515) | ropa (48,702) |
| Barranquilla | electronica (123,166) | **ropa** (33,464) | hogar (32,475) |
| Bucaramanga | electronica (84,621) | hogar (16,809) | ropa (16,010) |
| Cartagena | electronica (67,209) | hogar (13,175) | ropa (10,991) |
| Pereira | electronica (39,293) | hogar (11,379) | ropa (10,691) |

### Análisis

- **¿La categoría líder es la misma en todas las ciudades?** Sí: **electrónica** es la categoría #1 en las **7 ciudades**, sin excepción, y por un margen amplio (entre 3x y 4x la categoría que le sigue). Es la categoría que más rota en toda la red, independientemente de la región.

- **¿Hay diferencias regionales?** Sí, en el orden del #2 y #3: en la mayoría de ciudades el segundo lugar es **hogar** y el tercero **ropa**, pero **Barranquilla es la excepción**: allí **ropa** supera a **hogar** en el segundo puesto. Es una señal concreta de que el mix de inventario en la costa Caribe debería diferenciarse del resto del país.

- El hecho de que electrónica domine de forma tan uniforme sugiere que es una categoría de **alto valor unitario** más que de alta frecuencia — vale la pena cruzarlo con el número de transacciones por categoría (no solo el monto) para confirmar si el liderazgo es por precio o por volumen.

- **Implicación de negocio:** el inventario base de electrónica debe garantizarse en todas las tiendas por igual; hogar y ropa pueden ajustarse por región, dándole más peso a ropa en Barranquilla frente al estándar nacional (hogar > ropa).

## Cliente estrella por ciudad

¿Quién es el mejor cliente (mayor gasto total) en cada ciudad? Insumo para el programa de fidelización.

In [0]:
gasto_cliente_ciudad = (
    df.join(ciu, "id_ciudad")
      .groupBy("ciudad", "id_cliente")
      .agg(F.sum("monto").alias("gasto_cliente"))
)

promedio_ciudad = gasto_cliente_ciudad.groupBy("ciudad").agg(
    F.avg("gasto_cliente").alias("gasto_promedio_ciudad"),
    F.countDistinct("id_cliente").alias("clientes_unicos"),
)

w_ciudad_cliente = Window.partitionBy("ciudad").orderBy(F.desc("gasto_cliente"))

t3 = (
    gasto_cliente_ciudad
      .withColumn("rank", F.row_number().over(w_ciudad_cliente))
      .filter(F.col("rank") == 1)
      .join(promedio_ciudad, "ciudad")
      .withColumn("veces_el_promedio", F.round(F.col("gasto_cliente") / F.col("gasto_promedio_ciudad"), 2))
      .select("ciudad", "id_cliente", "gasto_cliente", "gasto_promedio_ciudad", "veces_el_promedio", "clientes_unicos")
      .orderBy(F.desc("gasto_cliente"))
)
t3.show(truncate=False)

+------------+----------+-----------------+---------------------+-----------------+---------------+
|ciudad      |id_cliente|gasto_cliente    |gasto_promedio_ciudad|veces_el_promedio|clientes_unicos|
+------------+----------+-----------------+---------------------+-----------------+---------------+
|Bogota      |106       |4422.549999999999|1219.2527875         |3.63             |800            |
|Medellin    |298       |4040.62          |1031.5944499999994   |3.92             |800            |
|Cali        |215       |3002.68          |538.0595897435899    |5.58             |780            |
|Bucaramanga |346       |2435.94          |261.1278782287822    |9.33             |542            |
|Cartagena   |617       |2034.95          |242.2486422413794    |8.4              |464            |
|Barranquilla|72        |1978.48          |327.66129078014194   |6.04             |705            |
|Pereira     |479       |1575.17          |197.89844444444432   |7.96             |405            |


### Resultados

| ciudad | cliente estrella | gasto total | promedio por cliente en la ciudad | veces el promedio |
|---|---:|---:|---:|---:|
| Bogota | 106 | 4,422.55 | 1,219.25 | 3.63x |
| Medellin | 298 | 4,040.62 | 1,031.59 | 3.92x |
| Cali | 215 | 3,002.68 | 538.06 | 5.58x |
| Bucaramanga | 346 | 2,435.94 | 261.13 | 9.33x |
| Cartagena | 617 | 2,034.95 | 242.25 | 8.40x |
| Barranquilla | 72 | 1,978.48 | 327.66 | 6.04x |
| Pereira | 479 | 1,575.17 | 197.90 | 7.96x |

In [0]:
t3_sql = spark.sql("""
    WITH gasto AS (
        SELECT c.ciudad, t.id_cliente, SUM(t.monto) AS gasto_cliente
        FROM transacciones t
        JOIN ciudades c ON t.id_ciudad = c.id_ciudad
        GROUP BY c.ciudad, t.id_cliente
    ),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY ciudad ORDER BY gasto_cliente DESC) AS rank
        FROM gasto
    ),
    promedio AS (
        SELECT ciudad, AVG(gasto_cliente) AS gasto_promedio_ciudad
        FROM gasto
        GROUP BY ciudad
    )
    SELECT r.ciudad, r.id_cliente, r.gasto_cliente, p.gasto_promedio_ciudad,
           ROUND(r.gasto_cliente / p.gasto_promedio_ciudad, 2) AS veces_el_promedio
    FROM ranked r
    JOIN promedio p ON r.ciudad = p.ciudad
    WHERE r.rank = 1
    ORDER BY r.gasto_cliente DESC
""")
t3_sql.show(truncate=False)

+------------+----------+-----------------+---------------------+-----------------+
|ciudad      |id_cliente|gasto_cliente    |gasto_promedio_ciudad|veces_el_promedio|
+------------+----------+-----------------+---------------------+-----------------+
|Bogota      |106       |4422.549999999999|1219.2527875         |3.63             |
|Medellin    |298       |4040.62          |1031.5944499999994   |3.92             |
|Cali        |215       |3002.68          |538.0595897435899    |5.58             |
|Bucaramanga |346       |2435.94          |261.1278782287822    |9.33             |
|Cartagena   |617       |2034.95          |242.2486422413794    |8.4              |
|Barranquilla|72        |1978.48          |327.66129078014194   |6.04             |
|Pereira     |479       |1575.17          |197.89844444444432   |7.96             |
+------------+----------+-----------------+---------------------+-----------------+



### Análisis

- **¿Cuánto gastó el mejor cliente frente al promedio de su ciudad?** Varía muchísimo según la plaza: en las ciudades grandes (Bogotá, Medellín) el cliente estrella gasta **~3.6–3.9 veces** el promedio de su ciudad, mientras que en ciudades pequeñas (Bucaramanga, Pereira, Cartagena) gasta **entre 7.9x y 9.3x** el promedio.

- Esto tiene una explicación estadística directa: las ciudades grandes tienen muchos más clientes únicos (≈800 en Bogotá/Medellín vs. ≈400–540 en las ciudades chicas), así que el gasto se reparte entre más personas y el promedio por cliente ya es alto por sí mismo; en las ciudades pequeñas hay menos clientes y un solo comprador fuerte destaca mucho más sobre el resto.

- **Implicación de negocio:** el programa de fidelización no debería usar un umbral de gasto único para todo el país. Un cliente que gasta ~$2.400 en Bucaramanga es proporcionalmente un cliente *tan estrella* (9.3x el promedio local) como uno que gasta ~$4.400 en Bogotá (3.6x el promedio local) — de hecho, en términos relativos, es un comprador **más atípico** para su mercado. Conviene definir el criterio de "cliente estrella" en términos relativos (múltiplo del promedio de su ciudad) y no en pesos absolutos.

- En términos absolutos, el cliente #106 en Bogotá es el mayor comprador individual de toda la cadena ($4,422.55 en el año), seguido del cliente #298 en Medellín — son los dos primeros candidatos naturales para un programa VIP nacional.

## Evolución mensual de ventas con acumulado

¿Cómo evolucionaron las ventas mes a mes y cuál es el acumulado del año? 

> Se extrae el mes con los primeros 7 caracteres de la fecha (`'2025-03'`), se agrupa y se usa `SUM(...) OVER (ORDER BY mes)` para el acumulado progresivo.

In [0]:
ventas_mes = (
    df.withColumn("mes", F.substring(F.col("fecha"), 1, 7))
      .groupBy("mes")
      .agg(F.sum("monto").alias("ventas_mes"))
)

w_acumulado = Window.orderBy("mes").rangeBetween(Window.unboundedPreceding, 0)

t4 = (
    ventas_mes
      .withColumn("acumulado", F.sum("ventas_mes").over(w_acumulado))
      .orderBy("mes")
)
t4.show(12, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------+------------------+------------------+
|mes    |ventas_mes        |acumulado         |
+-------+------------------+------------------+
|2025-01|253745.19000000015|253745.19000000015|
|2025-02|244188.97999999978|497934.1699999999 |
|2025-03|234270.4399999994 |732204.6099999993 |
|2025-04|227221.35000000018|959425.9599999995 |
|2025-05|216728.68000000025|1176154.6399999997|
|2025-06|220943.90000000005|1397098.5399999998|
|2025-07|228274.7299999997 |1625373.2699999996|
|2025-08|232584.26000000015|1857957.5299999998|
|2025-09|241525.3799999999 |2099482.9099999997|
|2025-10|222964.66999999987|2322447.5799999996|
|2025-11|233672.02999999965|2556119.6099999994|
|2025-12|229329.4199999997 |2785449.0299999993|
+-------+------------------+------------------+



In [0]:
t4_sql = spark.sql("""
    WITH ventas_mes AS (
        SELECT SUBSTRING(fecha, 1, 7) AS mes, SUM(monto) AS ventas_mes
        FROM transacciones
        GROUP BY SUBSTRING(fecha, 1, 7)
    )
    SELECT mes,
           ventas_mes,
           SUM(ventas_mes) OVER (ORDER BY mes) AS acumulado
    FROM ventas_mes
    ORDER BY mes
""")
t4_sql.show(12, truncate=False)

+-------+------------------+------------------+
|mes    |ventas_mes        |acumulado         |
+-------+------------------+------------------+
|2025-01|253745.19000000015|253745.19000000015|
|2025-02|244188.97999999978|497934.1699999999 |
|2025-03|234270.4399999994 |732204.6099999993 |
|2025-04|227221.35000000018|959425.9599999995 |
|2025-05|216728.68000000025|1176154.6399999997|
|2025-06|220943.90000000005|1397098.5399999998|
|2025-07|228274.7299999997 |1625373.2699999996|
|2025-08|232584.26000000015|1857957.5299999998|
|2025-09|241525.3799999999 |2099482.9099999997|
|2025-10|222964.66999999987|2322447.5799999996|
|2025-11|233672.02999999965|2556119.6099999994|
|2025-12|229329.4199999997 |2785449.0299999993|
+-------+------------------+------------------+



In [0]:
t4_pd = t4.toPandas()
display(t4_pd)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


mes,ventas_mes,acumulado
2025-01,253745.19000000015,253745.19000000015
2025-02,244188.97999999978,497934.1699999999
2025-03,234270.4399999994,732204.6099999993
2025-04,227221.35000000018,959425.9599999995
2025-05,216728.68000000025,1176154.6399999997
2025-06,220943.90000000005,1397098.5399999998
2025-07,228274.7299999997,1625373.2699999996
2025-08,232584.26000000015,1857957.5299999998
2025-09,241525.3799999999,2099482.9099999997
2025-10,222964.66999999987,2322447.5799999996


### Resultados

| mes | ventas_mes | acumulado |
|---|---:|---:|
| 2025-01 | 253,745.19 | 253,745.19 |
| 2025-02 | 244,188.98 | 497,934.17 |
| 2025-03 | 234,270.44 | 732,204.61 |
| 2025-04 | 227,221.35 | 959,425.96 |
| 2025-05 | 216,728.68 | **1,176,154.64** ← primer mes que supera 1M acumulado |
| 2025-06 | 220,943.90 | 1,397,098.54 |
| 2025-07 | 228,274.73 | 1,625,373.27 |
| 2025-08 | 232,584.26 | 1,857,957.53 |
| 2025-09 | 241,525.38 | 2,099,482.91 |
| 2025-10 | 222,964.67 | 2,322,447.58 |
| 2025-11 | 233,672.03 | 2,556,119.61 |
| 2025-12 | 229,329.42 | 2,785,449.03 |

### Análisis

- **¿Hay meses claramente más fuertes o débiles?** El patrón es de **declive suave desde enero hasta mayo** (de $253,745 a $216,729, una caída del 15%), seguido de una **recuperación gradual entre junio y septiembre** (hasta $241,525) y una **leve desaceleración en el último trimestre**. En general las ventas son bastante estables: el mes más fuerte (enero, $253,745) y el más débil (mayo, $216,729) difieren solo un **~17%**, sin picos ni caídas abruptas — no hay estacionalidad extrema tipo "diciembre dispara las ventas", como se ve en otros retailers.

- **¿Qué mes se superó el millón acumulado?** **Mayo de 2025**, con un acumulado de **$1,176,154.64** (el acumulado de abril era $959,425.96, por debajo del millón).

- El promedio mensual es de **~$232,121**; enero está claramente por encima de ese promedio (el mes más atípico al alza) y mayo por debajo (el más atípico a la baja).

- **Implicación de negocio:** con una demanda tan estable mes a mes, la planeación de inventario y de personal puede hacerse con proyecciones simples (sin necesidad de modelos estacionales complejos). Sí vale la pena investigar **por qué mayo es el mes más débil** — podría cruzarse con eventos externos (temporada de pagos, festivos, campañas de la competencia) para diseñar una promoción específica que sostenga las ventas en ese mes.

## Optimización

### 1. `cache()` — medir tiempo con y sin cache

La idea: `gasto_cliente_ciudad` se usa dos veces (una directa y una vía `promedio_ciudad`). Sin cache, Spark recalcula el join + agregación **cada vez** que se dispara una acción sobre ese DataFrame o sobre algo derivado de él. Con `cache()`, el resultado se materializa en memoria después de la primera acción y las siguientes lo reutilizan.

In [0]:
# --- SIN cache ---
gasto_sin_cache = (
    df.join(ciu, "id_ciudad")
      .groupBy("ciudad", "id_cliente")
      .agg(F.sum("monto").alias("gasto_cliente"))
)

t0 = time.time()
gasto_sin_cache.count()  # 1ra acción: fuerza el cálculo del join+groupBy
t1_ = time.time()
gasto_sin_cache.groupBy("ciudad").agg(F.avg("gasto_cliente")).collect()  # 2da acción: recalcula todo desde cero
t2_ = time.time()

print(f"1ra acción (sin cache): {t1_ - t0:.3f} s")
print(f"2da acción (sin cache, recalcula todo): {t2_ - t1_:.3f} s")

1ra acción (sin cache): 1.124 s
2da acción (sin cache, recalcula todo): 0.813 s


In [0]:
import time

gasto_con_cache = (
    df.join(ciu, "id_ciudad")
      .groupBy("ciudad", "id_cliente")
      .agg(F.sum("monto").alias("gasto_cliente"))
)

cache_soportado = True
try:
    gasto_con_cache = gasto_con_cache.cache()
    t0 = time.time()
    gasto_con_cache.count()
    t1_ = time.time()
    gasto_con_cache.groupBy("ciudad").agg(F.avg("gasto_cliente")).collect()
    t2_ = time.time()
except Exception as e:
    cache_soportado = False
    print("cache()/persist() no soportado en este compute:", e)
    t0 = time.time()
    gasto_con_cache.count()
    t1_ = time.time()
    gasto_con_cache.groupBy("ciudad").agg(F.avg("gasto_cliente")).collect()
    t2_ = time.time()

print(f"1ra acción: {t1_ - t0:.3f} s")
print(f"2da acción: {t2_ - t1_:.3f} s")

if not cache_soportado:
    print("\nNota: Databricks Serverless no soporta cache()/persist() explícito "
          "(PERSIST TABLE is not supported on serverless — SQLSTATE 0A000). "
          "Serverless administra su propio caché interno de forma automática y "
          "transparente. El comportamiento conceptual de cache() (evitar recalcular "
          "un DataFrame reutilizado) se documenta igual en el informe, aunque en este "
          "compute la ganancia no es medible explícitamente vía esta API.")
else:
    gasto_con_cache.unpersist()

cache()/persist() no soportado en este compute: [NOT_SUPPORTED_WITH_SERVERLESS] PERSIST TABLE is not supported on serverless compute. SQLSTATE: 0A000

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at com.databricks.serverless.ServerlessGCEdgeCheck$.throwError(ServerlessGCEdgeCheck.scala:72)
	at com.databricks.serverless.ServerlessGCEdgeCheck$.checkBlockCacheCommand(ServerlessGCEdgeCheck.scala:50)
	at org.apache.spark.sql.connect.service.SparkConnectAnalyzeHandler.$anonfun$process$44(SparkConnectAnalyzeHandler.scala:424)
	at com.databricks.util.LexicalThreadLocal$Handle.runWith(LexicalThreadLocal.scala:63)
	at org.apache.spark.sql.connect.execution.SparkConnectUdfRejectionScope$.withExecutionSideEffectScope(SparkConnectUdfRejectionScope.scala:31)
	at org.apache.spark.sql.connect.service.SparkConnectAnalyzeHandler.process(SparkConnectAnalyzeHandler.scala:379)
	at org.apache.spark.sql.connect.service.SparkConnectAnalyzeHandler.$anonfun$handle$5(SparkConnectAnalyzeHandler.scala:1

### 2. `explain()` — optimización de Catalyst

In [0]:
consulta_explain = (
    df.filter(F.col("monto") > 100)
      .join(ciu, "id_ciudad")
      .filter(F.col("departamento") == "Antioquia")
      .groupBy("ciudad")
      .agg(F.sum("monto").alias("ventas"))
)
consulta_explain.explain(True)

== Parsed Logical Plan ==
'Aggregate ['ciudad], ['ciudad, 'sum('monto) AS ventas#15707]
+- 'Filter '`==`('departamento, Antioquia)
   +- 'Join UsingJoin(Inner, [id_ciudad])
      :- 'Filter '`>`('monto, 100)
      :  +- 'UnresolvedRelation [analisis_clientes, default, transacciones], [], false
      +- 'UnresolvedRelation [analisis_clientes, default, ciudades], [], false

== Analyzed Logical Plan ==
ciudad: string, ventas: double
Aggregate [ciudad#15724], [ciudad#15724, sum(monto#15718) AS ventas#15707]
+- Filter (departamento#15725 = Antioquia)
   +- Project [id_ciudad#15716L, id_tx#15714L, id_cliente#15715L, categoria#15717, monto#15718, fecha#15719, ciudad#15724, departamento#15725]
      +- Join Inner, (id_ciudad#15716L = id_ciudad#15723L)
         :- Filter (monto#15718 > cast(100 as double))
         :  +- SubqueryAlias analisis_clientes.default.transacciones
         :     +- Relation analisis_clientes.default.transacciones[id_tx#15714L,id_cliente#15715L,id_ciudad#15716L,categor

**Qué buscar en el plan físico (`== Physical Plan ==`):**

- El filtro `monto > 100` está escrito **antes** del `join` en el código, y el filtro `departamento == 'Antioquia'` está escrito **después** del join. Sin embargo, en el plan físico ambos aparecen como `Filter` aplicados **justo después de leer cada fuente** (`FileScan csv ... PushedFilters: [...]`), es decir, Catalyst hace **predicate pushdown**: empuja el filtro de `departamento` hacia abajo, hasta el scan de `ciudades`, en vez de esperar a filtrar después del join. Esto reduce drásticamente el tamaño de las tablas *antes* de hacer el join, que es la operación más costosa.
- También se puede ver que Spark combina filtros consecutivos en un único nodo `Filter` (o los empuja como `PushedFilters` en el `FileScan`) en lugar de aplicar dos pasadas separadas sobre los datos — eso es la optimización de **filter pushdown / combinación de filtros** de Catalyst que pide el enunciado.
- Además, al ser `ciudades` una tabla de 7 filas, el plan debería mostrar un `BroadcastHashJoin` en lugar de un `SortMergeJoin` (ver punto 3 abajo).

### 3. Broadcast join — ¿dónde y por qué?

Usamos (implícita o explícitamente) un **broadcast join** en **todos los JOIN de T1, T2 y T3**, porque en los cuatro casos estamos uniendo `transacciones` (20.000 filas, la tabla "grande") con `ciudades` (7 filas, la tabla "pequeña").

- Spark tiene un umbral (`spark.sql.autoBroadcastJoinThreshold`, 10MB por defecto) por debajo del cual decide automáticamente enviar la tabla pequeña completa a cada executor (broadcast) en vez de hacer un shuffle de ambas tablas por la red (`SortMergeJoin`). Como `ciudades.csv` pesa unos pocos KB, cae naturalmente bajo ese umbral y Catalyst ya lo optimiza solo.
- Para dejarlo explícito (y que se note en el plan, o para forzarlo si el optimizador no lo detectara por algún motivo), se puede usar el hint `broadcast()`:

In [0]:
from pyspark.sql.functions import broadcast

t1_broadcast = (
    df.join(broadcast(ciu), "id_ciudad")
      .groupBy("departamento")
      .agg(F.sum("monto").alias("ventas_totales"))
)
t1_broadcast.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonGroupingAgg(keys=[departamento#15798], functions=[finalmerge_sum(merge sum#15813) AS sum(monto)#15803])
         +- PhotonShuffleExchangeSource false
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#24479]
               +- PhotonShuffleExchangeSink hashpartitioning(departamento#15798, 16)
                  +- PhotonGroupingAgg(keys=[departamento#15798], functions=[partial_sum(monto#15791) AS sum#15813])
                     +- PhotonProject [monto#15791, departamento#15798]
                        +- PhotonBroadcastHashJoin [id_ciudad#15789L], [id_ciudad#15796L], Inner, BuildRight, false, true, false
                           :- PhotonScan parquet analisis_clientes.default.transacciones[id_ciudad#15789L,monto#15791] DataFilters: [isnotnull(id_ciudad#15789L)], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1

**Por qué era el caso adecuado:** un `SortMergeJoin` requiere *shufflear* (repartir por la red) las dos tablas según la clave del join, lo que es costoso cuando la tabla grande tiene muchas filas. Con un broadcast join, en cambio, `ciudades` (7 filas) se copia completa a cada executor, y cada partición de `transacciones` se cruza localmente contra esa copia — se evita por completo el shuffle de la tabla grande, que es la parte cara de la operación. Como el catálogo de ciudades es minúsculo y estático, es exactamente el escenario de libro de texto para usar broadcast join.